# Age Group Detection — Model Training & Evaluation
A ResNet18 classifier trained on FairFace to predict one of 9 age brackets from a face image.


## 1. Load the Dataset
Download FairFace (padding=0.25 crop) from Hugging Face and inspect its structure.

In [18]:
!pip install -q datasets

from datasets import load_dataset

fairface = load_dataset("HuggingFaceM4/FairFace", "0.25")
print(fairface)
print(fairface["train"].features["age"])

README.md:   0%|          | 0.00/5.89k [00:00<?, ?B/s]

0.25/train-00000-of-00002-d405faba4f4b9b(…):   0%|          | 0.00/250M [00:00<?, ?B/s]

0.25/train-00001-of-00002-dd3cb681647274(…):   0%|          | 0.00/250M [00:00<?, ?B/s]

0.25/validation-00000-of-00001-951dbd63c(…):   0%|          | 0.00/63.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/86744 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10954 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'age', 'gender', 'race', 'service_test'],
        num_rows: 86744
    })
    validation: Dataset({
        features: ['image', 'age', 'gender', 'race', 'service_test'],
        num_rows: 10954
    })
})
ClassLabel(names=['0-2', '3-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', 'more than 70'])


## 2. Define Age Classes & Label Mapping
Use FairFace's native 9 age groups directly as our target classes.

In [19]:
# Target classes (all FairFace age groups)
classes = [
    "0-2",
    "3-9",
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "more than 70"
]

# Original FairFace age labels
fairface_age_names = fairface["train"].features["age"].names

# Map each FairFace label to a unique class
def map_age_label(age_idx):
    mapping = {
        "0-2": 0,
        "3-9": 1,
        "10-19": 2,
        "20-29": 3,
        "30-39": 4,
        "40-49": 5,
        "50-59": 6,
        "60-69": 7,
        "more than 70": 8,
    }
    return mapping[fairface_age_names[age_idx]]

# Add new label column
def add_new_label(example):
    example["label"] = map_age_label(example["age"])
    return example

# Use the complete dataset (no filtering)
train_raw = fairface["train"].map(add_new_label)
val_raw = fairface["validation"].map(add_new_label)

print(f"Train: {len(train_raw)} images")
print(f"Validation: {len(val_raw)} images")
print("Classes:", classes)


Map:   0%|          | 0/86744 [00:00<?, ? examples/s]

Map:   0%|          | 0/10954 [00:00<?, ? examples/s]

Train: 86744 images
Validation: 10954 images
Classes: ['0-2', '3-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', 'more than 70']


## 3. Check Class Balance
Count training samples per class — this reveals the imbalance that later needs weighted loss.

In [20]:
from collections import Counter

train_labels = train_raw["label"]
class_counts = Counter(train_labels)
class_counts_list = [class_counts[i] for i in range(9)]

for i, c in enumerate(classes):
    print(f"{c}: {class_counts_list[i]}")


0-2: 1792
3-9: 10408
10-19: 9103
20-29: 25598
30-39: 19250
40-49: 10744
50-59: 6228
60-69: 2779
more than 70: 842


## 4. Build Datasets & DataLoaders
Wrap the Hugging Face dataset in a PyTorch `Dataset`, apply augmentation/normalization, and create `train_loader` / `val_loader`.

In [21]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class FairFaceAgeDataset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.data = hf_dataset
        self.transform = transform
        self.labels = hf_dataset["label"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]["image"].convert("RGB")
        label = self.labels[idx]
        return self.transform(image), label

train_dataset = FairFaceAgeDataset(train_raw, train_transform)
val_dataset = FairFaceAgeDataset(val_raw, val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                           num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True, persistent_workers=True)

## 5. Define Model, Loss, and Optimizer
ResNet18 with a custom 9-class head, class-weighted `CrossEntropyLoss` to counter imbalance, Adam optimizer, and an LR scheduler.

In [ ]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = models.resnet18(weights="DEFAULT")
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 9)
)
model = model.to(device)

class_counts_list = [1792, 10408, 9103, 25598, 19250, 10744, 6228, 2779, 842]
weights_tensor = torch.tensor(class_counts_list, dtype=torch.float)
weights_tensor = weights_tensor.sum() / (len(weights_tensor) * weights_tensor)
weights_tensor = torch.clamp(weights_tensor, min=0.3, max=3.0)
criterion = nn.CrossEntropyLoss(weight=weights_tensor.to(device))

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

cuda


## 6. Train the Model
15 epochs, tracking train/validation loss and accuracy per epoch.

In [ ]:
from tqdm import tqdm
import torch

num_epochs = 15

for epoch in range(num_epochs):

    # =========================
    # Training
    # =========================
    model.train()

    running_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs} [Train]",
        leave=True
    )

    for images, labels in train_bar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (preds == labels).sum().item()

        train_acc = 100 * train_correct / train_total

        train_bar.set_postfix(
            Loss=f"{loss.item():.4f}",
            Acc=f"{train_acc:.2f}%"
        )

    train_loss = running_loss / len(train_loader)

    # =========================
    # Validation
    # =========================
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        val_bar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{num_epochs} [Val]",
            leave=False
        )

        for images, labels in val_bar:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

            val_acc = 100 * val_correct / val_total

            val_bar.set_postfix(
                Loss=f"{loss.item():.4f}",
                Acc=f"{val_acc:.2f}%"
            )

    val_loss /= len(val_loader)

    print(
        f"\nEpoch [{epoch+1}/{num_epochs}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_acc:.2f}% "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Acc: {val_acc:.2f}%"
    )

Epoch 1/15 [Train]: 100%|██████████| 1356/1356 [05:59<00:00,  3.77it/s, Acc=35.92%, Loss=1.0283]



Epoch [1/15] | Train Loss: 1.4610 | Train Acc: 35.92% | Val Loss: 1.2819 | Val Acc: 38.84%


Epoch 2/15 [Train]: 100%|██████████| 1356/1356 [05:54<00:00,  3.83it/s, Acc=43.56%, Loss=1.2468]



Epoch [2/15] | Train Loss: 1.2460 | Train Acc: 43.56% | Val Loss: 1.1696 | Val Acc: 48.74%


Epoch 3/15 [Train]: 100%|██████████| 1356/1356 [05:54<00:00,  3.83it/s, Acc=45.81%, Loss=1.2021]


...


Epoch [13/15] | Train Loss: 0.8664 | Train Acc: 57.06% | Val Loss: 1.1319 | Val Acc: 54.70%


Epoch 14/15 [Train]: 100%|██████████| 1356/1356 [05:49<00:00,  3.88it/s, Acc=57.98%, Loss=0.8421]



Epoch [14/15] | Train Loss: 0.8356 | Train Acc: 57.98% | Val Loss: 1.1415 | Val Acc: 49.98%


Epoch 15/15 [Train]: 100%|██████████| 1356/1356 [06:00<00:00,  3.76it/s, Acc=58.98%, Loss=0.7758]
                                                                                             


Epoch [15/15] | Train Loss: 0.8134 | Train Acc: 58.98% | Val Loss: 1.1628 | Val Acc: 55.08%


## 7. Per-Class Validation Accuracy
Break down accuracy by age bracket to check whether weighting actually helped the rare classes.

In [ ]:
from collections import defaultdict

model.eval()
class_correct = defaultdict(int)
class_total = defaultdict(int)

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        for label, pred in zip(labels, preds):
            class_total[label.item()] += 1
            if label.item() == pred.item():
                class_correct[label.item()] += 1

print(f"{'Class':<15}{'Accuracy':<10}{'Samples'}")
for i, name in enumerate(classes):
    total = class_total[i]
    correct = class_correct[i]
    acc = 100 * correct / total if total > 0 else 0
    print(f"{name:<15}{acc:.2f}%     {total}")

Class          Accuracy  Samples
0-2            80.90%     199
3-9            74.04%     1356
10-19          54.11%     1181
20-29          62.12%     3300
30-39          47.34%     2330
40-49          32.82%     1353
50-59          52.01%     796
60-69          57.63%     321
more than 70   27.97%     118


## 8. Save the Trained Model
Persist weights to `resnet18_age_detection.pth` for use in the Flask app or a fresh evaluation session.

In [ ]:
torch.save(model.state_dict(), "resnet18_age_detection.pth")
print("Saved.")

Saved.


## 9. Load Model for Standalone Evaluation
Reload the model from a saved `.pth` file — useful after a runtime restart, without retraining.

### 9.1 Upload Trained Weights

In [15]:
from google.colab import files
uploaded = files.upload()

Saving resnet18_age_detection.pth to resnet18_age_detection.pth


### 9.2 Rebuild Model Architecture & Load Weights

In [16]:
import torch
import torch.nn as nn
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

def load_model():
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, 9)
    )
    state_dict = torch.load(
        "resnet18_age_detection.pth",
        map_location="cpu",
        weights_only=True
    )
    model.load_state_dict(state_dict)
    model.eval()
    return model

model = load_model()
model = model.to(device)
print("Correct 9-class model loaded and moved to device.")

Using device: cuda

Correct 9-class model loaded and moved to device.


## 10. Final Evaluation Metrics
Accuracy, precision, recall, F1, classification report, and confusion matrix on the validation set.

In [22]:
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="weighted", zero_division=0))
print("Recall:", recall_score(y_true, y_pred, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_true, y_pred, average="weighted", zero_division=0))
print()
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))
print()
print("Confusion matrix (rows = true, columns = predicted):")
print(confusion_matrix(y_true, y_pred))

Accuracy: 0.5507577140770494
Precision: 0.5560404430298553
Recall: 0.5507577140770494
F1 Score: 0.5497010078840613

              precision    recall  f1-score   support

         0-2       0.62      0.81      0.70       199
         3-9       0.79      0.74      0.76      1356
       10-19       0.44      0.54      0.49      1181
       20-29       0.64      0.62      0.63      3300
       30-39       0.46      0.47      0.47      2330
       40-49       0.47      0.33      0.39      1353
       50-59       0.44      0.52      0.48       796
       60-69       0.43      0.58      0.49       321
more than 70       0.63      0.28      0.39       118

    accuracy                           0.55     10954
   macro avg       0.55      0.54      0.53     10954
weighted avg       0.56      0.55      0.55     10954


Confusion matrix (rows = true, columns = predicted):
[[ 161   34    1    1    1    0    0    1    0]
 [  98 1004  221   20    8    2    2    1    0]
 [   0  193  639  288   53   